In [1]:
import json
import os
os.makedirs("data/json_files", exist_ok=True)

In [2]:
# Sample nested JSON data
json_data = {
    "company": "TechCorp",
    "employees": [
        {
            "id": 1,
            "name": "John Doe",
            "role": "Software Engineer",
            "skills": ["Python", "JavaScript", "React"],
            "projects": [
                {"name": "RAG System", "status": "In Progress"},
                {"name": "Data Pipeline", "status": "Completed"}
            ]
        },
        {
            "id": 2,
            "name": "Jane Smith",
            "role": "Data Scientist",
            "skills": ["Python", "Machine Learning", "SQL"],
            "projects": [
                {"name": "ML Model", "status": "In Progress"},
                {"name": "Analytics Dashboard", "status": "Planning"}
            ]
        }
    ],
    "departments": {
        "engineering": {
            "head": "Mike Johnson",
            "budget": 1000000,
            "team_size": 25
        },
        "data_science": {
            "head": "Sarah Williams",
            "budget": 750000,
            "team_size": 15
        }
    }
}

In [3]:
json_data

{'company': 'TechCorp',
 'employees': [{'id': 1,
   'name': 'John Doe',
   'role': 'Software Engineer',
   'skills': ['Python', 'JavaScript', 'React'],
   'projects': [{'name': 'RAG System', 'status': 'In Progress'},
    {'name': 'Data Pipeline', 'status': 'Completed'}]},
  {'id': 2,
   'name': 'Jane Smith',
   'role': 'Data Scientist',
   'skills': ['Python', 'Machine Learning', 'SQL'],
   'projects': [{'name': 'ML Model', 'status': 'In Progress'},
    {'name': 'Analytics Dashboard', 'status': 'Planning'}]}],
 'departments': {'engineering': {'head': 'Mike Johnson',
   'budget': 1000000,
   'team_size': 25},
  'data_science': {'head': 'Sarah Williams',
   'budget': 750000,
   'team_size': 15}}}

In [4]:
with open('data/json_files/company_data.json', 'w') as f:
    json.dump(json_data, f, indent=2)

In [6]:
# Save JSON Lines format
jsonl_data = [
    {"timestamp": "2024-01-01", "event": "user_login", "user_id": 123},
    {"timestamp": "2024-01-01", "event": "page_view", "user_id": 123, "page": "/home"},
    {"timestamp": "2024-01-01", "event": "purchase", "user_id": 123, "amount": 99.99}
]

with open('data/json_files/events.jsonl', 'w') as f:
    for item in jsonl_data:
        f.write(json.dumps(item) + '\n')

### JSONLoader for Preprocessing

In [7]:
from langchain_community.document_loaders import JSONLoader

# Load the JSON file
json_loader = JSONLoader(
    file_path='data/json_files/company_data.json',
    jq_schema='.employees[]',  #jq query to extract employee details
    text_content=False # Get structured data (Full JSON Object) instead of text content

)

employee_docs = json_loader.load()
print(f"Loaded {len(employee_docs)} employee documents.")
print(employee_docs[0].page_content)  # Print the first employee's details

Loaded 2 employee documents.
{"id": 1, "name": "John Doe", "role": "Software Engineer", "skills": ["Python", "JavaScript", "React"], "projects": [{"name": "RAG System", "status": "In Progress"}, {"name": "Data Pipeline", "status": "Completed"}]}


In [10]:
print(type(employee_docs[0]))

<class 'langchain_core.documents.base.Document'>


## Intelligent Way

In [14]:
from langchain_core.documents import Document
from typing import List

def preprocess_json_intelligently(file_path: str) -> List[Document]:
    
    with open(file_path, 'r') as f:
        data = json.load(f)

        documents = []
        for employee in data.get('employees', []):
            content = f"""EmployeeProfile:
            Name : {employee['name']}
            Role : {employee['role']}
            Skills : {',' .join(employee['skills'])}

            Projects:"""

            for proj in employee.get('projects', []):
                content += f"\n- {proj['name']} (Status : {proj['status']}) "

        doc = Document(
            page_content=content,
            metadata={
                'source': file_path,
                'data_type': 'employee_profile',
                'employee_id': employee['id'],
                'employee_name': employee['name'],
                'role': employee['role']
            }
        )
        documents.append(doc)

    return documents


In [15]:
preprocess_json_intelligently('./data/json_files/company_data.json')

[Document(metadata={'source': './data/json_files/company_data.json', 'data_type': 'employee_profile', 'employee_id': 2, 'employee_name': 'Jane Smith', 'role': 'Data Scientist'}, page_content='EmployeeProfile:\n            Name : Jane Smith\n            Role : Data Scientist\n            Skills : Python,Machine Learning,SQL\n\n            Projects:\n- ML Model (Status : In Progress) \n- Analytics Dashboard (Status : Planning) ')]